# Session 2 · Grant it wrong, then fix it

Crown Street Markets, a fictional 40-store Charlotte grocery chain, gives its first analyst access to store sales. The loyalty table holds card numbers the analyst never needs.

In [ ]:
source ~/dsba6190-live-demo-02/env.sh && echo "$ANALYST" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · Crown Street's two datasets

In [ ]:
bq --project_id="$PROJECT" ls | grep crown_

In [ ]:
q "SELECT * FROM \`$RAW.loyalty_members\` LIMIT 3"

In [ ]:
n=$(as_analyst bq --project_id="$PROJECT" ls --format=json 2>/dev/null | jq length 2>/dev/null)
echo "datasets the analyst can see: ${n:-0}"

## Step 2 · How big each role is

In [ ]:
for r in roles/editor roles/bigquery.dataViewer roles/bigquery.jobUser; do
  printf '%-28s %5s permissions\n' $r $(gcloud iam roles describe $r --format=json | jq '.includedPermissions | length')
done

## Step 3 · Grant it wrong

In [ ]:
gcloud projects add-iam-policy-binding "$PROJECT" --member "serviceAccount:$ANALYST" \
  --role roles/editor --condition=None --quiet | grep -B1 -A2 cs-analyst

In [ ]:
S=$(date +%s); until as_analyst bq --project_id="$PROJECT" --location=US query --use_legacy_sql=false --format=pretty   "SELECT member_id, email, card_number FROM \`$RAW.loyalty_members\` LIMIT 3" >/dev/null 2>&1; do sleep 10; done
echo "took effect after $(( $(date +%s) - S )) s"

## Step 4 · What Editor lets the analyst do

In [ ]:
as_analyst bq --project_id="$PROJECT" --location=US query --use_legacy_sql=false --format=pretty \
  "SELECT member_id, email, card_number FROM \`$RAW.loyalty_members\` LIMIT 3"

In [ ]:
as_analyst bq --project_id="$PROJECT" rm -f -t "${RAW:?}.members_backup"

In [ ]:
bq --project_id="$PROJECT" ls "$PROJECT:$RAW"

In [ ]:
as_analyst gcloud compute instances list

## Step 5 · Fix it

In [ ]:
gcloud projects remove-iam-policy-binding "$PROJECT" --member "serviceAccount:$ANALYST" \
  --role roles/editor --condition=None --quiet >/dev/null && echo "removed roles/editor"
gcloud projects add-iam-policy-binding "$PROJECT" --member "serviceAccount:$ANALYST" \
  --role roles/bigquery.jobUser --condition=None --quiet >/dev/null && echo "granted roles/bigquery.jobUser"

In [ ]:
q "GRANT \`roles/bigquery.dataViewer\` ON SCHEMA \`$PROJECT.$CURATED\` TO 'serviceAccount:$ANALYST'"

## Step 6 · Read the policy

In [ ]:
gcloud projects get-iam-policy "$PROJECT" --flatten="bindings[].members" \
  --filter="bindings.members:$ANALYST" --format="table(bindings.role,bindings.members)"

In [ ]:
bq --project_id="$PROJECT" show --format=prettyjson "$PROJECT:$CURATED" | jq -c ".access[]"

## Step 7 · Nobody granted this one

In [ ]:
gcloud projects get-iam-policy "$PROJECT" --flatten="bindings[].members" \
  --filter="bindings.members:-compute@developer.gserviceaccount.com" --format="table(bindings.role,bindings.members)"

In [ ]:
gcloud iam service-accounts keys list --iam-account "$ANALYST" \
  --format="table(name.basename(),keyType,validAfterTime.date('%Y-%m-%d'))"

## Step 8 · A private network

In [ ]:
gcloud compute networks create "$NET" --subnet-mode custom

In [ ]:
gcloud compute networks subnets create "$SUBNET" --network "$NET" --region "$REGION" --range 10.20.0.0/24

In [ ]:
gcloud compute instances create "$VM" --zone "$ZONE" --machine-type e2-micro --subnet "$SUBNET" --no-address \
  --image-family debian-12 --image-project debian-cloud --shielded-secure-boot --labels course=dsba6190,session=02

In [ ]:
gcloud compute firewall-rules list --filter="network:$NET"

In [ ]:
until [ "$(gcloud compute instances describe "$VM" --zone "$ZONE" --format='value(status)')" = RUNNING ]; do sleep 5; done
sleep 45   # IAP needs the guest to finish booting before it can find the instance

In [ ]:
time timeout 60 gcloud compute ssh "$VM" --zone "$ZONE" --tunnel-through-iap --quiet --strict-host-key-checking=no --command hostname -- -o ConnectTimeout=20

## Step 9 · Open one door

In [ ]:
gcloud compute firewall-rules create "$FW" --network "$NET" --direction INGRESS --allow tcp:22 \
  --source-ranges 35.235.240.0/20

In [ ]:
cat probe.sh

In [ ]:
for i in 1 2 3 4 5 6; do
  timeout 120 gcloud compute ssh "$VM" --zone "$ZONE" --tunnel-through-iap --quiet --strict-host-key-checking=no --command 'bash -s' < probe.sh && break
  echo "retrying in 15 s"; sleep 15
done

## Step 10 · Private Google Access

In [ ]:
gcloud compute networks subnets update "$SUBNET" --region "$REGION" --enable-private-ip-google-access

In [ ]:
for i in 1 2 3 4 5 6; do
  timeout 120 gcloud compute ssh "$VM" --zone "$ZONE" --tunnel-through-iap --quiet --strict-host-key-checking=no --command 'bash -s' < probe.sh && break
  echo "retrying in 15 s"; sleep 15
done

## Step 11 · After: least privilege, scoped

In [ ]:
S=$(date +%s); while as_analyst bq --project_id="$PROJECT" --location=US query --use_legacy_sql=false --format=pretty   "SELECT member_id, email, card_number FROM \`$RAW.loyalty_members\` LIMIT 3" >/dev/null 2>&1; do sleep 10; done
echo "raw table refused; waited $(( $(date +%s) - S )) s more"

In [ ]:
as_analyst bq --project_id="$PROJECT" --location=US query --use_legacy_sql=false --format=pretty \
  "SELECT neighborhood, SUM(baskets) AS baskets, ROUND(SUM(revenue_usd)) AS revenue_usd
   FROM \`$CURATED.store_sales\` GROUP BY 1 ORDER BY 3 DESC LIMIT 5"

In [ ]:
as_analyst bq --project_id="$PROJECT" --location=US query --use_legacy_sql=false --format=pretty \
  "SELECT member_id, email, card_number FROM \`$RAW.loyalty_members\` LIMIT 3"

In [ ]:
as_analyst bq --project_id="$PROJECT" rm -f -t "${CURATED:?}.store_sales"

In [ ]:
as_analyst gcloud compute instances list

## Step 12 · Teardown

In [ ]:
gcloud compute instances delete "${VM:?}" --zone "${ZONE:?}" --quiet
gcloud compute firewall-rules delete "${FW:?}" --quiet
gcloud compute networks subnets delete "${SUBNET:?}" --region "${REGION:?}" --quiet
gcloud compute networks delete "${NET:?}" --quiet

In [ ]:
gcloud projects remove-iam-policy-binding "$PROJECT" --member "serviceAccount:${ANALYST:?}" \
  --role roles/bigquery.jobUser --condition=None --quiet >/dev/null && echo "removed jobUser"
gcloud iam service-accounts delete "${ANALYST:?}" --quiet
bq --project_id="$PROJECT" rm -r -f -d "${CURATED:?}"
bq --project_id="$PROJECT" rm -r -f -d "${RAW:?}"

In [ ]:
gcloud compute networks list --filter='name~crown-net'
gcloud iam service-accounts list --filter='email~cs-analyst'
bq --project_id="$PROJECT" ls | grep -c crown_ || true